In [1]:
import sys
sys.path.append('../src')
from preprocessing import full_pipeline

X_train, y_train, stats = full_pipeline('../data/train.csv')
X_test, _, _ = full_pipeline('../data/test.csv', stats=stats)

print(X_train.shape, y_train.shape)
print(X_test.shape)

(1460, 94) (1460,)
(1459, 94)


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
import numpy as np

from preprocessing import build_preprocessor, build_preprocessor_scaled


In [3]:
preprocessor = build_preprocessor(X_train)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

In [4]:
scores = cross_val_score(model_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
rmse_scores = -scores
print("RMSE per fold:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())

RMSE per fold: [0.13387191 0.16537198 0.16594884 0.11770477 0.19082157]
Mean RMSE: 0.1547438141437212


## KNN

In [5]:
from sklearn.neighbors import KNeighborsRegressor

knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', KNeighborsRegressor())
])

knn_scores = cross_val_score(knn_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
knn_rmse = -knn_scores
print("RMSE per fold:", knn_rmse)
print("Mean RMSE:", knn_rmse.mean())

RMSE per fold: [0.17038534 0.18485514 0.17543754 0.16011428 0.17981534]
Mean RMSE: 0.17412152954231896


## KNN

The `preprocessor` above only scales the numeric columns (including the ordinal-encoded ones,
like ExterQual_encoded), leaving nominal columns unscaled:
- Nominal columns: one-hot encoded, values are 0 or 1
- Numeric/ordinal columns: scaled via StandardScaler, values are approximately -3 to 3

In [6]:
preprocessor_scaled = build_preprocessor_scaled(X_train)

knn_pipeline_scaled = Pipeline(steps=[
    ('preprocessor', preprocessor_scaled),
    ('model', KNeighborsRegressor())
])

knn_scores_scaled = cross_val_score(knn_pipeline_scaled, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
knn_rmse_scaled = -knn_scores_scaled
print("RMSE per fold:", knn_rmse_scaled)
print("Mean RMSE:", knn_rmse_scaled.mean())

RMSE per fold: [0.20316563 0.20444215 0.1913099  0.17643792 0.19692406]
Mean RMSE: 0.19445593061296867


## Random Forest

In [7]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

rf_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
rf_rmse = -rf_scores
print("RMSE per fold:", rf_rmse)
print("Mean RMSE:", rf_rmse.mean())

RMSE per fold: [0.1334012  0.14813141 0.14453949 0.13346181 0.15208001]
Mean RMSE: 0.1423227846109297


## Gradient Boosting

In [8]:
from sklearn.ensemble import GradientBoostingRegressor

gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42))
])

gb_scores = cross_val_score(gb_pipeline, X_train, y_train, cv=5,scoring='neg_root_mean_squared_error')
gb_rmse = -gb_scores
print("RMSE per fold:", gb_rmse)
print("Mean RMSE:", gb_rmse.mean())

RMSE per fold: [0.11255176 0.13631043 0.12807356 0.11852529 0.12354213]
Mean RMSE: 0.12380063390492195


## Phase 5: Baseline Model Comparison

Built one Pipeline per model (preprocessor + model), evaluated all four with 
identical 5-fold cross-validation on log1p(SalePrice), scoring on RMSE.

| Model              | Mean RMSE | Fold spread     |
|---------------------|-----------|-----------------|
| Linear Regression    | 0.1547    | 0.118 - 0.191   |
| KNN                  | 0.1741    | 0.160 - 0.185   |
| Random Forest         | 0.1423    | 0.133 - 0.152   |
| Gradient Boosting     | 0.1236    | 0.113 - 0.136   |

**Winner: Gradient Boosting** - best mean RMSE and tightest fold spread
(most consistent across different data splits).

### Side experiment: does scaling one-hot columns help KNN?
Built a second preprocessor (build_preprocessor_scaled) that also scales the
one-hot encoded nominal columns (using StandardScaler(with_mean=False)), 
hypothesizing that the scale mismatch between 0/1 one-hot columns and 
-3 to 3 scaled numeric columns was hurting KNN's distance calculations.

Result: WORSE, not better (0.1945 vs 0.1741 baseline). 
Likely cause: with_mean=False divides by standard deviation without centering - 
for rare categories (e.g. Neighborhood values with only 2-3 houses), this 
produces very large scaled values instead of a tidy -3 to 3 range, actively 
distorting distance calculations for sparse categories rather than fixing them.
Conclusion: keep the original mixed-scaling preprocessor for all models going forward.

### Next steps
- Test is_outlier flag: compare performance with vs without the 2 flagged 
  outlier rows (523, 1298) from Phase 1/2
- Try xgboost/lightgbm as stronger gradient boosting alternatives
- Hyperparameter tuning on the best candidate(s)

## GB Using no outliers

- We remove two rows (we found in phase 1-2) that have very large GrLivArea, unusually low SalePrice,
both with SaleCondition = 'Partial' - unfinished new construction at time of sale.

- we expect gb to have a small improvement over using gb with them.

In [9]:
from preprocessing import load_data, flag_outliers

df_raw = load_data('../data/train.csv')
df_raw = flag_outliers(df_raw)
is_outlier_mask = df_raw['is_outlier']

In [10]:
X_train_no_outliers = X_train[~is_outlier_mask]
y_train_no_outliers = y_train[~is_outlier_mask]

gb_pipeline_no_outliers = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42))
])

gb_scores_no_outliers = cross_val_score(gb_pipeline_no_outliers, X_train_no_outliers, y_train_no_outliers, cv=5,scoring='neg_root_mean_squared_error')
gb_rmse_no_outliers = -gb_scores_no_outliers
print("RMSE per fold:", gb_rmse_no_outliers)
print("Mean RMSE:", gb_rmse_no_outliers.mean())

RMSE per fold: [0.11671862 0.12461112 0.13073093 0.11399376 0.12341435]
Mean RMSE: 0.12189375348297482


## Results

- Result: 0.1236 -> 0.1219, a small improvement, matching the prediction.

## RF Using no outliers

- Same logic with GB, since they are tree based algortihms we hypothesize we will have small improvements if we remove outliers

In [11]:
rf_pipeline_no_outliers = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

rf_scores_no_outliers = cross_val_score(rf_pipeline_no_outliers, X_train_no_outliers, y_train_no_outliers, cv=5, scoring='neg_root_mean_squared_error')
rf_rmse_no_outliers = -rf_scores_no_outliers
print("RMSE per fold:", rf_rmse_no_outliers)
print("Mean RMSE:", rf_rmse_no_outliers.mean())

RMSE per fold: [0.13282426 0.13448685 0.142881   0.1335671  0.13886919]
Mean RMSE: 0.13652568280160182


- Result: 0.1423 -> 0.1365, a modest improvement (~0.0058), slightly larger 
  than GB's (~0.0017) but still consistent with the tree-based robustness 
  hypothesis - both improved by a small, not dramatic, amount.

In [12]:
lr_pipeline_no_outliers = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_scores_no_outliers = cross_val_score(lr_pipeline_no_outliers, X_train_no_outliers, y_train_no_outliers,
                                          cv=5, scoring='neg_root_mean_squared_error')
lr_rmse_no_outliers = -lr_scores_no_outliers
print("Mean RMSE:", lr_rmse_no_outliers.mean())

Mean RMSE: 0.12805104369750325


In [13]:
knn_pipeline_no_outliers = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', KNeighborsRegressor())
])

knn_scores_no_outliers = cross_val_score(knn_pipeline_no_outliers, X_train_no_outliers, y_train_no_outliers,
                                           cv=5, scoring='neg_root_mean_squared_error')
knn_rmse_no_outliers = -knn_scores_no_outliers
print("Mean RMSE:", knn_rmse_no_outliers.mean())

Mean RMSE: 0.17013943349825664


## Final Results (with vs without outliers)

Format: with outliers vs without outliers (mean RMSE)

- `Linear Regression` : 0.1547 vs 0.1281 (largest improvement, ~0.0266)
- `KNN`               : 0.1741 vs 0.1701 (small improvement, ~0.0040; scaled-everything version was worse: 0.1944)
- `Random Forest`      : 0.1423 vs 0.1365 (modest improvement, ~0.0058)
- `Gradient Boosting`   : 0.1236 vs 0.1219 (smallest improvement, ~0.0017) - still overall best model

## Conclusion
All four models improved after removing the 2 flagged outlier rows, with 
no exceptions - Linear Regression's original counter-prediction (that it 
would perform worse without them) didn't hold. Improvement size correlates 
with how globally sensitive each model is to individual data points: 
Linear Regression (fits one equation across all data) improved the most, 
Gradient Boosting (tree-based, isolates outliers into their own branches) 
improved the least - consistent with the original theory tested earlier.

Decision: drop the 2 outlier rows going forward for all subsequent modeling.

In [14]:
from xgboost import XGBRegressor

xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(random_state = 42))
])

xgb_scores = cross_val_score(xgb_pipeline, X_train_no_outliers, y_train_no_outliers,
                               cv=5, scoring='neg_root_mean_squared_error')
xgb_rmse = -xgb_scores
print("RMSE per fold:", xgb_rmse)
print("Mean RMSE:", xgb_rmse.mean())

RMSE per fold: [0.12888791 0.13733684 0.14136752 0.13035695 0.13132809]
Mean RMSE: 0.13385546404755722


In [15]:
from lightgbm import LGBMRegressor

lgbm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LGBMRegressor(random_state=42))
])

lgbm_scores = cross_val_score(lgbm_pipeline, X_train_no_outliers, y_train_no_outliers,
                                cv=5, scoring='neg_root_mean_squared_error')
lgbm_rmse = -lgbm_scores
print("RMSE per fold:", lgbm_rmse)
print("Mean RMSE:", lgbm_rmse.mean())

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001282 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3963
[LightGBM] [Info] Number of data points in the train set: 1166, number of used features: 168
[LightGBM] [Info] Start training from score 12.021352
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001091 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3994
[LightGBM] [Info] Number of data points in the train set: 1166, number of used features: 165
[LightGBM] [Info] Start training from score 12.023516
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001215 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3983
[LightGBM] [Info] Number of data points in the train set: 1166, number of used features: 168
[LightGBM] [Info] Start t

- `XGBoost` : 0.1338
- `LightGBM` : 0.1252

# Top 3 Models for Hyperparameter testing
- `Gradient Boosting`
- `LightGBM`


In [16]:
gb_pipeline.named_steps['model'].get_params()

{'alpha': 0.9,
 'ccp_alpha': 0.0,
 'criterion': 'deprecated',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'squared_error',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': 42,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [17]:
from sklearn.model_selection import GridSearchCV

gb_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth':[3, 5, 10],
    'model__learning_rate':[0.1, 0.2, 0.3],
    'model__subsample':[0.8, 0.9, 1.0]
}

gb_grid_search = GridSearchCV(gb_pipeline_no_outliers, gb_param_grid, cv = 5, scoring='neg_root_mean_squared_error')
gb_grid_search.fit(X_train_no_outliers, y_train_no_outliers)

print("Best RMSE:", -gb_grid_search.best_score_)
print("Best parameters:", gb_grid_search.best_params_)

Best RMSE: 0.11733114812374687
Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200, 'model__subsample': 0.8}


In [23]:
gb_pipeline_tuned = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        random_state=42
    ))
])

gb_pipeline_tuned.fit(X_train_no_outliers, y_train_no_outliers)

test_predictions_log = gb_pipeline_tuned.predict(X_test)
test_predictions = np.expm1(test_predictions_log)

In [24]:
print(X_test.isnull().sum()[X_test.isnull().sum() > 0])

Series([], dtype: int64)


In [25]:
print(test_predictions[:10])
print(test_predictions.min(), test_predictions.max(), test_predictions.mean())

[130374.04329338 164611.11509864 181613.83038651 189068.83398099
 190984.45199689 175174.81336875 178289.50555964 166739.85895939
 186720.96927083 122703.53340625]
34542.589279594955 737170.8898794353 178354.69344695


In [26]:
lgbm_pipeline_no_outliers = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LGBMRegressor(random_state=42, verbose=-1))
])

lgbm_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 10],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__subsample': [0.8, 0.9, 1.0]
}

lgbm_grid_search = GridSearchCV(
    lgbm_pipeline_no_outliers,
    lgbm_param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

lgbm_grid_search.fit(X_train_no_outliers, y_train_no_outliers)

print("Best RMSE:", -lgbm_grid_search.best_score_)
print("Best parameters:", lgbm_grid_search.best_params_)

Best RMSE: 0.11987997398379405
Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200, 'model__subsample': 0.8}


In [27]:
lgbm_pipeline_tuned = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LGBMRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        random_state=42,
        verbose=-1
    ))
])

lgbm_pipeline_tuned.fit(X_train_no_outliers, y_train_no_outliers)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](94,)","['MSSubClass','MSZoning','LotFrontage',...,'HasMasVnr','IsNew', 'IsPartialSale']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,94
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehot', ...), ('scale', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This sub

In [28]:
test_predictions_lgbm_log = lgbm_pipeline_tuned.predict(X_test)
test_predictions_lgbm = np.expm1(test_predictions_lgbm_log)

print(test_predictions_lgbm[:10])
print(test_predictions_lgbm.min(), test_predictions_lgbm.max(), test_predictions_lgbm.mean())

[121966.32999671 164188.34369546 178646.91342429 193426.14256083
 190876.35663027 175401.82571276 179412.69653447 169474.36031219
 184617.46932467 124875.73157079]
49640.111995036226 631073.5556782143 178564.0984672152


In [33]:
import pandas as pd

test_ids = pd.read_csv('../data/test.csv')['Id']

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_predictions
})

submission.to_csv('../submissions/gb_tuned_submission.csv', index=False)
submission.head()

,Id,SalePrice
0,1461,130374.043293
1,1462,164611.115099
2,1463,181613.830387
3,1464,189068.833981
4,1465,190984.451997


In [34]:
print(submission.shape)          
print(submission.isnull().sum())

(1459, 2)
Id           0
SalePrice    0
dtype: int64
